In [8]:
import urllib.request

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
urllib.request.urlretrieve(url, "yellow_tripdata_2024-01.parquet")
print("Descargado")

Descargado


In [9]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import os

# Utilizamos esta lina para configurar la variable de entorno JAVA_HOME y asegurarnos de que Spark pueda encontrar Java
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"

# Agregamos Java al PATH para que Spark pueda ejecutarlo
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Configuramos Spark para permitir el uso del Security Manager de Java
spark = (SparkSession.builder
         .appName("nyc-taxi")
         .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
         .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")
         .getOrCreate())

# Leemos el archivo Parquet utilizando Spark
df = spark.read.parquet("yellow_tripdata_2024-01.parquet")

Transformacion en SQL:
SELECT
    VendorID,
    payment_type,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(SUM(total_amount), 2) AS ingreso_total,
    ROUND(AVG(tip_amount), 2) AS propina_promedio
FROM yellow_taxi
WHERE trip_distance > 0
    AND total_amount > 0
    AND passenger_count BETWEEN 1 AND 6
GROUP BY VendorID, payment_type
HAVING COUNT(*) > 100
ORDER BY ingreso_total DESC

In [ ]:
result_df = (
    
df


  .filter((F.col("trip_distance") > 0) &
         (F.col("total_amount") > 0) &
         (F.col("passenger_count") > 0) 
    )

  .groupBy("VendorID") 

  .agg(
        F.count("*").alias("cantidad_viajes"),
        F.round(F.avg("trip_distance"), 2).alias("distancia_promedio"),
        F.round(F.sum("total_amount"), 2).alias("ingreso_total"),
        F.round(F.avg("tip_amount"), 2).alias("propina_promedio")
    )  

  .withcolumn("") 
    
)


result_df.show()


+--------+---------------+------------------+-------------+----------------+
|VendorID|cantidad_viajes|distancia_promedio|ingreso_total|propina_promedio|
+--------+---------------+------------------+-------------+----------------+
|       1|         638784|              3.14|1.656839938E7|            3.16|
|       2|        2085233|              3.35|5.809688905E7|            3.57|
+--------+---------------+------------------+-------------+----------------+



In [17]:

resultado = (
    df
    # WHERE trip_distance > 0 AND total_amount > 0
    #       AND passenger_count BETWEEN 1 AND 6
    .filter(
        (F.col("trip_distance") > 0) &
        (F.col("total_amount") > 0) &
        (F.col("passenger_count").between(1, 6))
    )

    # GROUP BY VendorID, payment_type
    .groupBy("VendorID", "payment_type")

    # SELECT con las agregaciones + alias
    .agg(
        F.count("*").alias("cantidad_viajes"),
        F.round(F.avg("trip_distance"), 2).alias("distancia_promedio"),
        F.round(F.sum("total_amount"), 2).alias("ingreso_total"),
        F.round(F.avg("tip_amount"), 2).alias("propina_promedio")
    )

    # HAVING COUNT(*) > 100
    .filter(F.col("cantidad_viajes") > 100)

    # ORDER BY ingreso_total DESC
    .orderBy(F.col("ingreso_total").desc())
)

resultado.show()

+--------+------------+---------------+------------------+-------------+----------------+
|VendorID|payment_type|cantidad_viajes|distancia_promedio|ingreso_total|propina_promedio|
+--------+------------+---------------+------------------+-------------+----------------+
|       2|           1|        1740016|              3.33|4.962187485E7|            4.27|
|       1|           1|         533539|              3.18| 1.42655531E7|            3.78|
|       2|           2|         320838|              3.45|   7868733.72|             0.0|
|       1|           2|          97079|              2.91|   2115858.08|             0.0|
|       2|           4|          19955|               3.4|    501706.85|             0.0|
|       1|           3|           5472|              3.01|    119927.64|             0.0|
|       2|           3|           4382|               2.7|    100432.32|            0.01|
|       1|           4|           2693|              3.68|     67042.11|             0.0|
+--------+

In [ ]:
df_1 = df.select()

In [ ]:
df = spark.read.parquet("yellow_tripdata_2024-01.parquet")
df.show(5)   # esta acción SÍ dispara la lectura — debería responder en segundos

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

Ejercicio 2:

SELECT
    VendorID,
    payment_type,
    CASE
        WHEN trip_distance < 2 THEN 'Corto'
        WHEN trip_distance < 10 THEN 'Medio'
        ELSE 'Largo'
    END AS tipo_viaje,
    COUNT(*) AS cantidad_viajes,
    ROUND(AVG(trip_distance), 2) AS distancia_promedio,
    ROUND(SUM(total_amount), 2) AS ingreso_total,
    ROUND(AVG(tip_amount), 2) AS propina_promedio,
    ROUND(AVG(tip_amount) / NULLIF(AVG(fare_amount), 0) * 100, 1) AS pct_propina,
    MAX(total_amount) AS viaje_mas_caro
FROM yellow_taxi
WHERE trip_distance > 0
    AND total_amount > 0
    AND fare_amount > 0
    AND passenger_count BETWEEN 1 AND 6
    AND payment_type IN (1, 2)
GROUP BY
    VendorID,
    payment_type,
    CASE
        WHEN trip_distance < 2 THEN 'Corto'
        WHEN trip_distance < 10 THEN 'Medio'
        ELSE 'Largo'
    END
HAVING COUNT(*) > 50
    AND AVG(total_amount) > 10
ORDER BY ingreso_total DESC, cantidad_viajes DESC
LIMIT 20

In [ ]:
result = (
    
    df

    .agg(F.count("*").alias("cantidad_viajes"),
         F.round(F.avg("trip_distance"),2).alias("distancia_p romedio"),
         F.round(F.sum("total_amount"), 2).alias("ingreso_total"),
         F.round(F.avg("tip_amount"), 2).alias("propina_promedio"),
         F.round(F.avg("tip_amount") / F.fillna((F.avg("total_amount")) * 100, 1).alias("porcentaje_propina")),
         F.round((F.max("total_amount").alias("viaje_mas_caro")))         
         )
            
    .withColumn("tipo_viaje", 
         F.when(F.col("trip_distance") < 2, "Corto")       
          .when(F.col("trip_distance") < 10, "Medio")
          .otherwise ("Largo")           
         )        
            
     .filter(
         F.col(("trip_distance") > 0) &
         F.col(("total_amount") > 0) &
         F.col(("fare_amount") > 0) &
         F.col("passenger_count").between(1, 6) &
         F.col("payment_type").isin(1, 2, 3, 4)
         )

     .groupBy("VendorID", "payment_type","tipo_viaje") 

     .filter((F.count("*") > 100) & 
              (F.avg("total_amount") > 10)
            )

     .orderBy(F.col("ingreso_total").desc(),
              F.col("cantidad_viajes").desc()            
  
              )

     .limit(20)   

     )          
            


